# 01. Dataset, split, dan tokenisasi

Tujuan: memahami perubahan dari kode website mentah menjadi batch next-token untuk Mini GPT.

**Urutan belajar:** sumber dan storage -> sampel -> normalisasi -> deduplikasi/family split -> BPE -> label -> antrean SFT.

Jalankan sel berurutan dengan kernel **Mini GPT WebCodeGen (Python 3.11 CUDA)**.
Notebook membaca artifact lokal dan menjalankan contoh kecil di memori. Download dan preparation tersedia sebagai command yang dijalankan secara sadar.
Output tersimpan berasal dari eksekusi nyata pada 12 Oktober 2026. Setelah clone, raw data/tokenizer/checkpoint perlu disiapkan kembali karena tidak masuk Git.

In [1]:
from pathlib import Path
import sys, json, shutil, datetime
import pandas as pd
import torch
from IPython.display import display, Markdown, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/model.json").is_file() and (p / "src/models/minigpt.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Buka notebook dari folder proyek ini.")
if Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    raise RuntimeError("Pilih kernel Mini GPT WebCodeGen dari .venv proyek ini.")
sys.path.insert(0, str(ROOT))
def load_json(relative):
    return json.loads((ROOT / relative).read_text(encoding="utf-8"))
print("Interpreter:", sys.executable)
print("Waktu eksekusi:", datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).isoformat())
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Interpreter: F:\Generative_AI\Vibe_Coding_Proyek_1\.venv\Scripts\python.exe
Waktu eksekusi: 2026-10-12T22:17:02.857559+07:00
PyTorch: 2.14.1+cu130 | CUDA: 13.0
GPU: NVIDIA GeForce RTX 5070


## 1. Sumber dan batas storage

Hanya kolom teks diunduh. Screenshot dan gambar tidak diperlukan untuk target prompt-ke-PageSpec.
Tabel berikut dibaca dari config dan manifest, termasuk revision serta lisensi.
Cadangan disk dibandingkan dengan ruang bebas saat notebook dijalankan, bukan ruang bebas historis saat download.

In [2]:
config = load_json("configs/dataset.json")
summary = load_json("data/manifests/pipeline_summary.json")
display(pd.DataFrame(summary["sources"]))
free = shutil.disk_usage(ROOT).free
reserve = config.get("reserve_free_bytes", 15 * 1024**3)
print(f"Ruang bebas sekarang: {free / 1024**3:.2f} GiB")
print(f"Cadangan config: {reserve / 1024**3:.2f} GiB; di atas cadangan: {free >= reserve}")
print(f"Raw historis: {summary['raw_rows']:,} baris, {summary['raw_bytes'] / 1024**2:.2f} MiB")

,id,revision,license,rows,bytes
0,websight_v01,b11f8172f89c992b56ac702319e02c428cca4a4e,cc-by-4.0,20000,40723783
1,websight_v02,b11f8172f89c992b56ac702319e02c428cca4a4e,cc-by-4.0,30000,72847730
2,vulcan,19d2a482c78554e59feca0ce2b50fb7835e90c6f,mit,5000,9705334


Ruang bebas sekarang: 25.34 GiB
Cadangan config: 15.00 GiB; di atas cadangan: True
Raw historis: 55,000 baris, 117.57 MiB


## 2. Baca satu sampel lokal

`idea` merupakan deskripsi sumber, sedangkan `text` berisi kode. Deskripsi WebSight belum menjadi instruksi Indonesia yang ditinjau.
Kode ditampilkan sebagai teks agar bisa dipelajari. Lisensi/atribusi asal tetap melekat.

In [3]:
from itertools import islice
from src.data.normalize import read_jsonl, normalize_source_code
raw_path = ROOT / "data/raw/websight_v02-30000.jsonl"
sample = next(read_jsonl(raw_path)) if raw_path.exists() else None
if sample:
    display(pd.DataFrame([{k: sample[k] for k in ["source_id", "source_row_id", "license_id", "style_mode", "idea"]}]))
    print(sample["text"][:1600])
else:
    print("Raw lokal belum ada. Lihat command download pada bagian terakhir.")

,source_id,source_row_id,license_id,style_mode,idea
0,websight_v02,v0.2:b11f8172f89c992b56ac702319e02c428cca4a4e:...,cc-by-4.0,tailwind,"Real Estate Company Site: A centered logo, a s..."


<html>
<link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
<body class="bg-gray-100">
  <div class="container mx-auto px-4 py-8">
    <div class="flex justify-center">
      <img src="https://source.unsplash.com/random/300x200/?real-estate" alt="Real Estate Logo" class="h-16">
    </div>
    <div class="mt-8">
      <p class="text-center text-xl">Welcome to our Real Estate Company Site, your one-stop destination for all your real estate needs. We specialize in finding the perfect home for you, whether it's a luxury estate or a cozy cottage. Our team of experienced realtors is dedicated to providing you with the best service and the most accurate information. Whether you're looking for a new home, selling your current one, or investing in real estate, we're here to help.</p>
    </div>
    <div class="mt-8">
      <form class="flex justify-center">
        <input type="text" placeholder="Search for properties" class="px-4 py-2 rounded-l">

## 3. Normalisasi dan family template

Normalizer mempertahankan HTML, CSS, dan JavaScript standalone. Fragmen HTML bisa dibungkus root/body.
Family dihitung dari struktur, lalu seluruh family ditempatkan pada satu partition.
Ini heuristik, sehingga nol overlap hash/family tidak membuktikan nol kesamaan semantik.

In [4]:
if sample:
    normalized, language, transform, family = normalize_source_code(sample["text"])
    display(pd.DataFrame([{"language": language, "transform": transform,
                           "family": family, "bytes": len(normalized.encode("utf-8"))}]))
display(pd.DataFrame([{"tahap": "Raw", "baris": summary["raw_rows"]},
                      {"tahap": "Diterima", "baris": summary["pretrain"]["accepted"]},
                      {"tahap": "Unik", "baris": summary["split"]["deduplicated_rows"]}]))
print("Ditolak:", summary["pretrain"]["rejected"], "| Duplikat:", summary["split"]["duplicates"])
display(pd.DataFrame(summary["token_caches"]).T[["rows", "tokens", "blocks", "block_size"]])

,language,transform,family,bytes
0,html,none,layout-76baa911fc71057c31ebe538,1148


,tahap,baris
0,Raw,55000
1,Diterima,54996
2,Unik,51211


Ditolak: 4 | Duplikat: 3785


,rows,tokens,blocks,block_size
train,41639,19038770,18593,1024
validation,5016,2266877,2214,1024
test,4556,2074052,2026,1024


## 4. Lihat isolasi partition aktual

Rasio 80/10/10 berlaku pada group, sehingga rasio jumlah baris dapat berbeda.
Alokasi family pretraining juga menjadi acuan SFT. Baris sumber yang sama tidak boleh berpindah ke partition berbeda saat dianotasi.
Sel ini hanya membaca file dan menghitung overlap ulang.

In [5]:
partitions = {}
for split in ["train", "validation", "test"]:
    path = ROOT / f"data/processed/pretrain/{split}.jsonl"
    if path.exists():
        groups, hashes, rows = set(), set(), 0
        for row in read_jsonl(path):
            groups.add(row["group_id"])
            hashes.add(row["normalized_text_sha256"])
            rows += 1
        partitions[split] = (groups, hashes, rows)
overlaps = []
for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    if left in partitions and right in partitions:
        overlaps.append({"pasangan": f"{left} / {right}",
                         "family_overlap": len(partitions[left][0] & partitions[right][0]),
                         "exact_overlap": len(partitions[left][1] & partitions[right][1])})
display(pd.DataFrame(overlaps))
if not overlaps:
    print("Partition lokal belum ada; jumlah historis tersedia pada manifest di atas.")

,pasangan,family_overlap,exact_overlap
0,train / validation,0,0
1,train / test,0,0
2,validation / test,0,0


## 5. BPE proyek dan label next-token

Vocabulary 8.192 dilatih hanya pada train partition. Tidak ada tokenizer pralatih yang diunduh.
`input_ids[t]` memprediksi `labels[t]`, yaitu token berikutnya. Loss tidak melakukan shift kedua.
Padding memakai label `-100` agar tidak dihitung dalam loss.

In [6]:
from tokenizers import Tokenizer
from src.data.dataset import encode_pretraining, PretrainingBlockDataset
from src.data.sources import sha256_file
tokenizer_path = ROOT / "data/tokenizer/tokenizer.json"
tokenizer = Tokenizer.from_file(str(tokenizer_path)) if tokenizer_path.exists() else None
if tokenizer:
    print("Tokenizer cocok manifest:", sha256_file(tokenizer_path) == summary["tokenizer"]["tokenizer_sha256"])
    text = "<h1>Halo dunia</h1>"
    inputs, labels = encode_pretraining(text, tokenizer)
    display(pd.DataFrame({"input_id": inputs, "label_id": labels,
                          "input_token": [tokenizer.id_to_token(i) for i in inputs],
                          "label_token": [tokenizer.id_to_token(i) for i in labels]}))
    token_file = ROOT / "data/processed/pretrain/train.tokens.bin"
    if token_file.exists():
        dataset = PretrainingBlockDataset(token_file, 1024, tokenizer.token_to_id("<pad>"))
        batch = dataset[0]
        print("Blok:", len(dataset), "| Bentuk:", {k: tuple(v.shape) for k, v in batch.items()})
        print("Label valid blok pertama:", int((batch["labels"] != -100).sum()))
else:
    print("Tokenizer lokal belum ada. Preparation melatih BPE baru dari train.")

Tokenizer cocok manifest: True


,input_id,label_id,input_token,label_token
0,2,33,<bos>,<
1,33,77,<,h
2,77,22,h,1
3,22,35,1,>
4,35,45,>,H
5,45,326,H,al
6,326,84,al,o
7,84,713,o,Ġd
8,713,488,Ġd,un
9,488,1204,un,ia


Blok: 18593 | Bentuk: {'input_ids': (1024,), 'labels': (1024,)}
Label valid blok pertama: 1024


## 6. Antrean instruksi dan keputusan SFT

Pretraining mengajarkan pola kode. Untuk produk, target SFT adalah JSON PageSpec, dengan prompt Indonesia dan katalog interaksi yang terbatas.
`draft` belum boleh digunakan sebagai data reviewed. Review mencakup kesesuaian prompt, layout, teks, theme, interaksi, lisensi, dan sumber family.

In [7]:
display(pd.DataFrame([summary["sft"]]))
queue = ROOT / "data/annotations/review_queue.jsonl"
if queue.exists():
    drafts = list(islice(read_jsonl(queue), 3))
    display(pd.DataFrame([{k: r[k] for k in ["source_row_id", "prompt", "prompt_language", "annotation_status", "reviewer"]} for r in drafts]))
    print(json.dumps(drafts[0]["target_spec"], ensure_ascii=False, indent=2)[:1700])

,status,draft_queue_rows,reviewed_production_rows
0,awaiting_review,119,0


,source_row_id,prompt,prompt_language,annotation_status,reviewer
0,v0.1:b11f8172f89c992b56ac702319e02c428cca4a4e:...,,en,draft,
1,v0.1:b11f8172f89c992b56ac702319e02c428cca4a4e:...,,en,draft,
2,v0.1:b11f8172f89c992b56ac702319e02c428cca4a4e:...,,en,draft,


{
  "schema_version": "1.0",
  "title": "Website",
  "language": "en",
  "style_mode": "plain_css",
  "theme": {
    "primary": "#2563eb",
    "background": "#ffffff",
    "text": "#111827",
    "font": "sans"
  },
  "sections": [
    {
      "id": "hero-1",
      "type": "hero",
      "props": {
        "heading": "Website",
        "text": "Our Nonprofit Mission Services Mission At Our Nonprofit, we strive to make a difference in the world. Through our services, we aim to promote equality and inclusivity across all communities. Services We provide several services that aim to improve the lives of those in need. Our services include but are not limited to, food distribution, shelter provision, and community program management. Copyright © Our Nonprofit 2022"
      }
    }
  ],
  "interactions": []
}


## 7. Menjalankan preparation secara terpisah

Command berikut dijalankan dari root proyek. Downloader memakai ulang file complete yang checksum-nya cocok.
Preparation harus mempertahankan tokenizer dan split yang sama ketika hanya menambahkan anotasi.

```powershell
.\.venv\Scripts\python.exe qa/run_logged.py --label download_dataset -- .\.venv\Scripts\python.exe -m src.data.sources --config configs/dataset.json
.\.venv\Scripts\python.exe qa/run_logged.py --label prepare_dataset -- .\.venv\Scripts\python.exe -m src.data.pipeline --config configs/dataset.json --train-tokenizer
```

Lanjut ke [02_model_dan_training.ipynb](02_model_dan_training.ipynb).
Rujukan proyek: [panduan data](../docs/PIPELINE_DATA_GUIDE.md), [atribusi dataset](../data/licenses/README.md).